# 06 · Real pilot quality, effort and precision planning

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Requires the approved real pilot; synthetic outputs never supply sample-size evidence.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Read actual pilot records and independent reviews

In [ ]:
from oncoplate.governance import require_gate
require_gate(cfg,'new_collection')
pilot_root=Path(cfg['root'])/'data/pilot'
records=read_table(pilot_root/'records.csv');reviews=read_table(pilot_root/'reviews.csv')
print('Records:',len(records),'Review rows:',len(reviews))

## 2. Assess annotation quality and workload

In [ ]:
from oncoplate.studies import pilot_quality
report=pilot_quality(records,reviews)
write_json(p['reports']/"pilot_quality.json",report);print(json.dumps(report,indent=2))

## 3. Run pilot-based paired-group precision scenarios
This is a planning projection, not proof of clinical safety or a completed power analysis. Candidate comparisons must come from genuinely evaluated pilot methods.

In [ ]:
from oncoplate.statistics import pilot_precision
pilot_effects=read_table(pilot_root/'paired_group_differences.csv')
assert {'group_id','risk_difference'}.issubset(pilot_effects)
assert not pilot_effects.group_id.duplicated().any()
scenario=pilot_precision(pilot_effects.risk_difference.to_numpy(float),n_groups=(40,60,80,120),B=1000)
print(scenario);write_table(p['reports']/"pilot_precision_scenarios.csv",scenario)

## 4. Record a real pilot decision
The next benchmark size and effect/coverage margins require review before scale-up.

In [ ]:
pilot_decision={'status':'pending_review','pilot_records':len(records),'target_main_records':2400,'target_external_records':400,
'required_changes':[],'review_reference':'','test_outcomes_seen':False}
destination=p['private']/"pilot_decision.json"
if not destination.exists():write_json(destination,pilot_decision)
print(destination)

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
